In [19]:
import numpy as np
import pandas as pd

# ----------------------------
# Functions
# ----------------------------

def research(x):
    return 200_000 * np.log(1 + x) / np.log(101)

def scale(y):
    return 7 * y / 100

def get_speed_multiplier(my_z, others):
    all_speeds = others + [my_z]
    unique_sorted = sorted(set(all_speeds), reverse=True)

    rank = unique_sorted.index(my_z) + 1
    n = len(unique_sorted)

    if n == 1:
        return 0.9

    return 0.9 - (rank - 1) * (0.8 / (n - 1))


def pnl(x, y, z, others):
    speed_mult = get_speed_multiplier(z, others)
    gross = research(x) * scale(y) * speed_mult
    cost = x + y + z
    return gross - cost


# ----------------------------
# Grid search
# ----------------------------

def run_grid(others):
    results = []

    for x in range(101):
        for y in range(101 - x):
            for z in range(101 - x - y):
                val = pnl(x, y, z, others)
                results.append((x, y, z, x+y+z, val))

    df = pd.DataFrame(results, columns=[
        "research", "scale", "speed", "budget_used", "pnl"
    ])
    return df.sort_values("pnl", ascending=False).reset_index(drop=True)


# ----------------------------
# Try multiple scenarios
# ----------------------------

scenarios = {
    "balanced_players": [20, 40, 50, 60, 70, 30, 45],
    "low_speed_players": [10, 20, 30, 25, 15, 35, 40],
    "high_speed_players": [60, 70, 80, 90, 75, 65, 85],
}

all_results = []

for name, others in scenarios.items():
    df = run_grid(others)
    df["scenario"] = name
    print(f"\n=== Top results for {name} ===")
    print(df.head(10))
    all_results.append(df.head(50))  # keep top 50 for merging

# ----------------------------
# Combine for robustness
# ----------------------------

combined = pd.concat(all_results)

grouped = combined.groupby(["research", "scale", "speed", "budget_used"]).agg(
    avg_pnl=("pnl", "mean"),
    min_pnl=("pnl", "min")
).reset_index()

print("\n=== Robust (best avg PnL) ===")
print(grouped.sort_values("avg_pnl", ascending=False).head(20))

print("\n=== Robust (best worst-case PnL) ===")
print(grouped.sort_values("min_pnl", ascending=False).head(20))


=== Top results for balanced_players ===
   research  scale  speed  budget_used            pnl          scenario
0        13     36     51          100  193406.756125  balanced_players
1        12     37     51          100  193197.094468  balanced_players
2        14     35     51          100  192949.891744  balanced_players
3        11     38     51          100  192226.213815  balanced_players
4        15     34     51          100  191903.513753  balanced_players
5        10     39     51          100  190375.720796  balanced_players
6        16     33     51          100  190331.165583  balanced_players
7        17     32     51          100  188285.941733  balanced_players
8        13     35     51           99  188032.568455  balanced_players
9        13     35     52          100  188031.568455  balanced_players

=== Top results for low_speed_players ===
   research  scale  speed  budget_used            pnl           scenario
0        15     45     40          100  340532.015

np.float64(-4.61512051684126)

In [13]:
def func_log(x):
    
    return 200000 * (np.log(1+x) / np.log(101))

func_log(0)


np.float64(0.0)

In [15]:
func_log(10)

np.float64(103914.74129648815)

In [16]:
func_log(20)

np.float64(131936.85523979316)

In [17]:
func_log(30)

np.float64(148814.62756840335)

In [20]:
func_log(50)

np.float64(170388.86063219846)

In [12]:
func_log(5)

np.float64(0.38823676709842325)

In [51]:
b_list = [x for x in range(50,75)]

def generate_pairs(b_list):
    
    results = []
    
    for b in b_list:
        
        for x in range(b+1):
            
            y = b - x
            
            r = generate_research(y)
            s = generate_scale(x)
            
            #print(f"b: {b}, x: {x}, y: {y}")
            
            mul = s * r
            #print(f"mul: {mul}")
            
            results.append((b, x, y, mul))
    
    results_df = pd.DataFrame(results, columns=["b", "x", "y", "mul"])
    return results_df
            
def generate_research(a):
    return 200000 * (np.log(1+a) / np.log(101))

def generate_scale(a):
    return 7 * a / 100
    

In [52]:
results_df = generate_pairs(b_list)
results_df

,b,x,y,mul
0,50,0,50,0.000000
1,50,1,49,11867.148837
2,50,2,48,23611.727570
3,50,3,47,35229.945104
4,50,4,46,46717.797490
...,...,...,...,...
1570,74,70,4,341756.872530
1571,74,71,3,298578.680649
1572,74,72,2,239950.654146
1573,74,73,1,153494.673855


In [53]:
results_df.sort_values("mul", ascending=False)

,b,x,y,mul
1556,74,56,18,500190.656180
1555,74,55,19,499816.601152
1557,74,57,17,499773.874677
1554,74,54,20,498721.312806
1558,74,58,16,498485.191660
...,...,...,...,...
869,64,64,0,0.000000
805,64,0,64,0.000000
804,63,63,0,0.000000
741,63,0,63,0.000000


In [59]:
best_per_budget = results_df.loc[results_df.groupby("b")["mul"].idxmax()]

print(best_per_budget[["b", "x", "y", "mul"]].to_string(index=False))
print()
print(f"Research range: {best_per_budget['y'].min()} – {best_per_budget['y'].max()}")
print(f"Scale range:    {best_per_budget['x'].min()} – {best_per_budget['x'].max()}")

 b  x  y           mul
50 37 13 296207.150334
51 38 13 304212.748991
52 39 13 312218.347649
53 39 14 320380.671405
54 40 14 328595.560416
55 41 14 336810.449426
56 42 14 345025.338436
57 42 15 353248.016542
58 43 15 361658.683603
59 44 15 370069.350663
60 45 15 378480.017724
61 46 15 386890.684784
62 46 16 395350.324420
63 47 16 403944.896690
64 48 16 412539.468960
65 49 16 421134.041230
66 50 16 429728.613500
67 50 17 438398.135681
68 51 17 447166.098395
69 52 17 455934.061108
70 53 17 464702.023822
71 54 17 473469.986536
72 54 18 482326.704173
73 55 18 491258.680176
74 56 18 500190.656180

Research range: 13 – 18
Scale range:    37 – 56


In [58]:
x

,b,x,y,mul
1556,74,56,18,500190.656180
1555,74,55,19,499816.601152
1557,74,57,17,499773.874677
1554,74,54,20,498721.312806
1558,74,58,16,498485.191660
...,...,...,...,...
1503,74,3,71,38919.888732
1502,74,2,72,26030.276764
1501,74,1,73,13056.411222
1500,74,0,74,0.000000


In [44]:
results_df[results_df["b"] == 50].sort_values("mul", ascending=False)

,b,x,y,mul
37,50,37,13,296207.150334
36,50,36,14,295736.004374
38,50,38,12,295670.081245
35,50,35,15,294373.347118
39,50,39,11,293981.278676
34,50,34,16,292215.457180
40,50,40,10,290961.275630
33,50,33,17,289342.769550
41,50,41,9,286381.219852
32,50,32,18,285823.232103


In [45]:
results_df[results_df["b"] == 55].sort_values("mul", ascending=False)

,b,x,y,mul
92,55,41,14,336810.449426
91,55,40,15,336426.682421
93,55,42,13,336235.143622
90,55,39,16,335188.318530
94,55,43,12,334574.039303
89,55,38,17,333182.583118
95,55,44,11,331671.186198
88,55,37,18,330483.112119
96,55,45,10,327331.435084
87,55,36,19,327152.684391


In [46]:
results_df[results_df["b"] == 60].sort_values("mul", ascending=False)

,b,x,y,mul
152,60,45,15,378480.017724
151,60,44,16,378161.179880
153,60,46,14,377884.894478
150,60,43,17,377022.396686
154,60,47,13,376263.136911
...,...,...,...,...
110,60,3,57,36952.145848
109,60,2,58,24738.476062
108,60,1,59,12420.222541
107,60,0,60,0.000000


In [47]:
results_df[results_df["b"] == 65].sort_values("mul", ascending=False)

,b,x,y,mul
217,65,49,16,421134.041230
216,65,48,17,420862.210254
218,65,50,15,420533.353026
215,65,47,18,419802.872151
219,65,51,14,418959.339530
...,...,...,...,...
171,65,3,62,37704.683523
170,65,2,63,25232.001182
169,65,1,64,12663.032648
168,65,0,65,0.000000


In [36]:
def res_scale(x, y):
    
    r = generate_research(x)
    s = generate_scale(y)
    
    mul = s * r
    return mul

In [37]:
res_scale(20, 40)

np.float64(369423.19467142085)

In [38]:
res_scale(25, 35)

np.float64(345921.0431893989)

In [39]:
res_scale(30, 30)

np.float64(312510.71789364703)

In [40]:
res_scale(15, 45)

np.float64(378480.01772369357)

In [48]:
res_scale(33, 37)

np.float64(395797.84430015145)

In [50]:
res_scale(17, 53)

np.float64(464702.0238220836)